# Calculations and demos of fields and particles

In [ ]:
import numpy as np
from pathlib import Path

import seaborn as sns
import matplotlib.pyplot as plt

from scipy.interpolate import CubicSpline, RegularGridInterpolator

import sys
root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from stepsic.random import RNG
from stepsic.parameters import CosmoParameters
from stepsic.field import cubic_voxels, fourier_grid
from stepsic.cosmology import ColossusCosmology, CAMBCosmology

In [ ]:
import logging
log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
# Facecolor values from S. Conradi @S_Conradi/@profConradi
custom_settings = {
    'figure.facecolor': '#f4f0e8',
    'axes.facecolor': '#f4f0e8',
    'axes.edgecolor': '0.3',
    'axes.linewidth' : '0.5',
    'axes.grid': False,
    'grid.color': '0.7',
    'grid.linestyle': ':',
    'grid.alpha': 0.6,
    'xtick.bottom': True,
    'xtick.top': True,
    'ytick.left': True,
    'ytick.right': True,
}
for t in ['xtick', 'ytick']:
    custom_settings[f'{t}.direction'] = 'in'
    custom_settings[f'{t}.color'] = '0.3'
    for m in ['major', 'minor']:
        custom_settings[f'{t}.{m}.width'] = 0.5
        custom_settings[f'{t}.{m}.size'] = 6 if m == 'major' else 3
sns.set_theme(palette=sns.color_palette('deep', as_cmap=False),
              rc=custom_settings)
plt.rcParams['text.usetex'] = False

In [ ]:
params = CosmoParameters(path=Path('..', 'config.toml')).get_parameters()

In [ ]:
rng = RNG(seed=params['SEED'])

## White noise generation

### Generate uniformly distributed particles in a box

In [ ]:
def create_particles(npart: int, Lbox, seed=None):
    r'''
    Create a set of particles uniformly distributed in a cubic box.

    Parameters
    ----------
    npart : int
        The number of particles to create.
    Lbox : float or list of float
        Length of the box in each dimension [Lx, Ly, Lz] or a single
        float value for a cubic box.
    seed : int, optional
        Random seed for reproducibility.

    Returns
    -------
    particles : ndarray of shape (npart, 3)
        Particle positions in the simulation box.
    '''
    return (rng.uniform(size=(npart, 3), seed=seed) - 0.5) * np.array(Lbox)

In [ ]:
x = create_particles(params['NPART'], params['LBOX'], seed=params['SEED'])
x

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*6, nr*4), dpi=120)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]
    ax.scatter(*x[:, idx].T, color='0.3', s=1**2, ec='none', alpha=0.5)
    ax.set_xlim(-params['LBOX'][idx[0]]//2, params['LBOX'][idx[0]]//2)
    ax.set_ylim(-params['LBOX'][idx[1]]//2, params['LBOX'][idx[1]]//2)
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')

plt.show()

### Designate grid cells for the density field

In [ ]:
def create_grid(nvox, dk):
    '''
    Create a regular grid for the simulation box.

    Parameters
    ----------
    nvox : tuple of int
        The number of voxels in each dimension of the simulation box.
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.

    Returns
    -------
    coords : ndarray of shape (N, 3)
        The coordinates of the grid points in the simulation box.
    grid : ndarray of shape (3, Nx, Ny, Nz)
        The grid coordinates in each dimension, where Nx, Ny, Nz are the
        number of voxels in each dimension.
    '''
    mesh = [np.arange(-(n-1)*dk/2, n*dk/2, dk) for n in nvox]
    xx, yy, zz = np.meshgrid(*mesh, indexing='ij')
    coords = np.stack([xx.ravel(), yy.ravel(), zz.ravel()], axis=1)
    return coords, np.array((xx, yy, zz))

In [ ]:
nvox, dk = cubic_voxels(params['NMESH'], params['LBOX'])
coords, grid = create_grid(nvox, dk)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*6, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)
    
    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]

    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s = x1[slicer], x2[slicer]

    ax.scatter(x1_s, x2_s, color='0.3', s=2**2, ec='none')
    ax.set_xlim(-params['LBOX'][idx[0]]//2, params['LBOX'][idx[0]]//2)
    ax.set_ylim(-params['LBOX'][idx[1]]//2, params['LBOX'][idx[1]]//2)
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*6, nr*4), dpi=120)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)
    
    idx = [k for k in range(3) if k != i]
    x1, x2 = coords.T[idx[0]], coords.T[idx[1]]
    
    ax.scatter(x1, x2, color='0.3', s=2**2, ec='none')
    ax.set_xlim(-params['LBOX'][idx[0]]//2, params['LBOX'][idx[0]]//2)
    ax.set_ylim(-params['LBOX'][idx[1]]//2, params['LBOX'][idx[1]]//2)
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

### Generate a Gaussian white noise that obeys cosmological expectations

#### Imaginary space generation

In [ ]:
def white_noise_k(nvox, seed=None):
    r'''
    Return a complex Gaussian array W(k) on the ``rfftn()`` grid
    `(Nx, Ny, Nz//2+1)`, obeying Hermitian constraints that guarantee
    :math:`\delta(x)` reconstructed with ``irfftn()`` is real.

    The field is generated in the imaginary space.

    Parameters
    ----------
    nvox : tuple of int
        The number of voxels in each dimension of the grid (Nx, Ny, Nz).
    seed : int
        Random seed for reproducibility. If None, a random seed is used.
    '''
    shape = (nvox[0], nvox[1], nvox[2]//2 + 1)

    # Independent N(0,1) draws for real and imaginary parts
    rng = RNG(seed=seed)
    a = rng.normal(std=0.5**0.5, size=shape)
    b = rng.normal(std=0.5**0.5, size=shape)
    w_k = a + 1j * b

    # Enforce Hermitian constraints W(k) = W*(-k)
    w_k[0, 0, 0] = 0.0  # set DC=0 (mean density) as we only need fluctuations

    # 1. Nyquist frequencies for even-sized dimensions must be real
    # 2. kz = 0 plane must be real
    if nvox[0] % 2 == 0:
        w_k[nvox[0]//2, 0, 0] = w_k[nvox[0]//2, 0, 0].real
        w_k[nvox[0]//2, :, :] = w_k[nvox[0]//2, :, :].real
    if nvox[1] % 2 == 0:
        w_k[0, nvox[1]//2, 0] = w_k[0, nvox[1]//2, 0].real
        w_k[:, nvox[1]//2, :] = w_k[:, nvox[1]//2, :].real
    if nvox[2] % 2 == 0:
        w_k[:, :, nvox[2]//2] = w_k[:, :, nvox[2]//2].real
    
    if nvox[0] % 2 == 0 and nvox[1] % 2 == 0:
        w_k[nvox[0]//2, nvox[1]//2, 0] = w_k[nvox[0]//2, nvox[1]//2, 0].real
    
    return w_k

In [ ]:
nvox, _ = cubic_voxels(params['NMESH'], params['LBOX'])

field_k = white_noise_k(nvox)
field = np.fft.irfftn(field_k)
log.info(f'Field shape: {field.shape}')
f_mean, f_std, f_median = np.mean(field), np.std(field), np.median(field)
log.info(f'Field: mean={f_mean:.3f}, std={f_std:.3f}, median={f_median:.3f}')

#### Real space generation

In [ ]:
def white_noise(nvox, seed=None):
    r'''
    Return a complex Gaussian array W(k) on the ``rfftn()`` grid
    `(Nx, Ny, Nz//2+1)`, obeying Hermitian constraints that guarantee
    :math:`\delta(x)` reconstructed with ``irfftn()`` is real.

    The field is generated in the real space.

    Parameters
    ----------
    nvox : tuple of int
        Number of voxels in each dimension (Nx, Ny, Nz).
    dk : float
        Step size in each dimension.
    seed : int or None, optional
        Random seed for reproducibility. If None, uses the default RNG.

    Returns
    -------
    w_k : ndarray
        3D array of white noise values.
    '''
    w_k = np.fft.rfftn(rng.normal(size=nvox, seed=seed))
    w_k[0, 0, 0] = 0.0  # set DC=0 (mean density) as we only need fluctuations
    return w_k

In [ ]:
nvox, dk = cubic_voxels(params['NMESH'], params['LBOX'])
grid = np.meshgrid(*(np.arange(-(n-1)*dk/2, n*dk/2, dk) for n in nvox), indexing='ij')

field_k = white_noise(nvox, seed=params['SEED'])
field = np.fft.irfftn(field_k)
log.info(f'Field shape: {field.shape}')
f_mean, f_std, f_median = np.mean(field), np.std(field), np.median(field)
log.info(f'Field: mean={f_mean:.3f}, std={f_std:.3f}, median={f_median:.3f}')

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)

    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]
    
    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]

    ax.scatter(x1_s, x2_s, c=field_s, s=4**2, ec='none')
    ax.set_xlim(-params['LBOX'][idx[0]]//2, params['LBOX'][idx[0]]//2)
    ax.set_ylim(-params['LBOX'][idx[1]]//2, params['LBOX'][idx[1]]//2)
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*6, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)

    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]

    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]

    ax.pcolormesh(x1_s, x2_s, field_s, shading='auto')
    ax.set_xlim(-params['LBOX'][idx[0]]//2, params['LBOX'][idx[0]]//2)
    ax.set_ylim(-params['LBOX'][idx[1]]//2, params['LBOX'][idx[1]]//2)
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    ax.set_title(f'{labels[idx[0]]}{labels[idx[1]]} slice at {labels[i]}={c_i[i]}')
plt.show()

### Generate a Gaussian white noise that obeys cosmological expectations in real space

In [ ]:
def generate_delta_k(kh, pk, nvox, dk, *, field=None, seed=None):
    r'''
    Generates the Fourier modes of an arbitrary input field from a
    given power spectrum.

    Parameters
    ----------
    kh : ndarray
        1D array of wavenumbers `k`, in [Mpc].
    pk : ndarray
        1D array of the matter power spectrum `P(k)` at the initial redshift.
    nvox : tuple of int
        The number of voxels in each dimension of the grid `(Nx, Ny, Nz)`.
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    field : ndarray
        Fourier transform of the real-valued overdensity field
        :math:`\delta(\mathbf{x})` defined on a regular grid, where
        :math:`\mathbf{x}` are the comoving coordinates.
    seed : int
        The seed for the random number generator.

    Returns
    -------
    delta_k : ndarray
        A 3D complex-valued array of shape `(Nx, Ny, Nz//2+1)` representing
        the Fourier modes of the overdensity field.
    '''
    _, kmod = fourier_grid(nvox, dk, hermitian=True)
    
    # interpolate the power spectrum in log-log space
    spline = CubicSpline(np.log(kh), np.log(pk), extrapolate=True)
    pk_grid = np.zeros_like(kmod, dtype=float)
    mask = kmod > 0
    if np.any(mask):
        ktarget_log = np.log(kmod[mask])
        pk_grid[mask] = np.exp(spline(ktarget_log))

    if field is None:
        field = white_noise(nvox=nvox, seed=seed)

    # Sirko 2005; Bagla & Padmanabhan 1997; Klypin & Holtzman 1997
    return field * np.sqrt(pk_grid / dk**3)

In [ ]:
kmin = 1/np.min(params['LBOX'])
kmax = 1.0
npoints = 2048

cosmo_colossus = ColossusCosmology(
    H0=params['H0'], Om0=params['OMEGA_M'], Ob0=params['OMEGA_B'],
    Ol0=params['OMEGA_L'], sigma8=params['SIGMA8'], ns=params['NS'],
    Neff=params['NNU'], w0=params['W0'], wa=params['WA'], Tcmb0=1e-6)
g1 = 1
D1 = g1 * cosmo_colossus.Dzplus0(params['REDSHIFT'])
g2 = - 3.0/7.0 * params['OMEGA_M']**(-1/143)
D2 = g2 * D1**2  # Bernardeau et al. 2002, eq. 97  # Unused!
log.info(f"D1(z={params['REDSHIFT']}) = {D1:.3e}")
log.info(f"D2(z={params['REDSHIFT']}) = {D2:.3e}")

cosmo_camb = CAMBCosmology(
    H0=params['H0'], ombh2=params['OMBH2'], omch2=params['OMCH2'],
    omk=params.get('OMK', 0.0), mnu=params['MNU'], nnu=params['NNU'],
    YHe=params['YHE'], TCMB=params['TCMB'], zrei=params['ZREI'],
    w0=params['W0'], wa=params['WA'], nonlinear=False)
kh, pk, pk3 = cosmo_camb.get_spectrum(
    z=0, As=params['AS'], ns=params['NS'], sigma8_init=params['SIGMA8'],
    kmin=kmin, kmax=kmax, npoints=npoints)
pk = pk[0]*D1**2

In [ ]:
nvox, dk = cubic_voxels(params['NMESH'], params['LBOX'])
grid = np.meshgrid(*(np.arange(-(n-1)*dk/2, n*dk/2, dk) for n in nvox), indexing='ij')
field = white_noise(nvox=nvox, seed=params['SEED'])
delta_k = generate_delta_k(kh, pk, nvox, dk, field=field)
delta_x = np.fft.irfftn(delta_k)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*6, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)

    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]

    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, delta_xs = x1[slicer], x2[slicer], delta_x[slicer]

    ax.pcolormesh(x1_s, x2_s, delta_xs, shading='auto')
    ax.set_xlim(-params['LBOX'][idx[0]]//2, params['LBOX'][idx[0]]//2)
    ax.set_ylim(-params['LBOX'][idx[1]]//2, params['LBOX'][idx[1]]//2)
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    ax.set_title(f'{labels[idx[0]]}{labels[idx[1]]} slice at {labels[i]}={c_i[i]}')
plt.show()

### Field interpolation to particle positions

In [ ]:
def interpolate_field(x, field, dk, method='linear'):
    r'''
    Interpolate a grid-based field onto particle positions using periodic
    boundaries.

    Parameters
    ----------
    x : ndarray of shape (N, 3)
        Particle positions in the simulation box.
    field : ndarray
        The grid-based field (e.g. a displacement field) defined on a
        regular grid.
    dk : float
        The uniform step size in each dimension, calculated as the length
        of the shortest dimension divided by the number of voxels in
        that dimension.
    method : str, optional
        The interpolation method to use. This can be 'linear', 'nearest',
        or 'cubic'. The default is 'linear'.

    Returns
    -------
    interp_values : ndarray of shape (N,)
        Field values interpolated at the particle positions.
    '''
    nvox = field.shape
    mesh = (np.arange(-(n-1)*dk/2, n*dk/2, dk) for n in nvox)
    interpolator = RegularGridInterpolator(
        mesh,
        field,
        method=method,
        bounds_error=False,
        fill_value=None  # Extrapolate using periodic wrapping if needed
    )
    return interpolator(x)

In [ ]:
nvox, dk = cubic_voxels(params['NMESH'], params['LBOX'])
grid = np.meshgrid(*(np.arange(-(n-1)*dk/2, n*dk/2, dk) for n in nvox), indexing='ij')

field = rng.normal(size=nvox)

x = create_particles(32**3, params['LBOX'], seed=params['SEED'])
field_interp = interpolate_field(x, field, dk)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*6, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)

    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]
    
    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]

    ax.scatter(x1_s, x2_s, c=field_s, s=4**2, ec='none')
    ax.set_xlim(-params['LBOX'][idx[0]]//2, params['LBOX'][idx[0]]//2)
    ax.set_ylim(-params['LBOX'][idx[1]]//2, params['LBOX'][idx[1]]//2)
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    ax.set_title(f'{labels[idx[0]]}{labels[idx[1]]} slice at {labels[i]}={c_i[i]}')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*6, nr*4), dpi=120)

labels = ['x', 'y', 'z']
c_i = [int(ni/2) for ni in nvox]  # index of the center slice along each axis

for i, ax in enumerate(axes.flat):
    ax.set_aspect(1)
    
    idx = [k for k in range(3) if k != i]
    x1, x2 = grid[idx[0]], grid[idx[1]]

    slicer = [slice(None)] * 3
    slicer[i] = c_i[i]
    slicer = tuple(slicer)
    x1_s, x2_s, field_s = x1[slicer], x2[slicer], field[slicer]

    ax.pcolormesh(x1_s, x2_s, field_s, shading='auto')
    ax.set_xlim(-params['LBOX'][idx[0]]//2, params['LBOX'][idx[0]]//2)
    ax.set_ylim(-params['LBOX'][idx[1]]//2, params['LBOX'][idx[1]]//2)
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
    ax.set_title(f'{labels[idx[0]]}{labels[idx[1]]} slice at {labels[i]}={c_i[i]}')
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*6, nr*4), dpi=120)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]
    ax.scatter(*x[:, idx].T, c=field_interp, s=1**2, ec='none', alpha=0.5)
    ax.set_xlim(-params['LBOX'][idx[0]]//2, params['LBOX'][idx[0]]//2)
    ax.set_ylim(-params['LBOX'][idx[1]]//2, params['LBOX'][idx[1]]//2)
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()

### Periodicity

In [ ]:
periodic = [0, 0, 1]
Lbox = [100, 100, 50]

x_pert = 20
x = rng.uniform(size=(2000, 3)) * (np.array(Lbox)+x_pert) - x_pert/2
x = np.where(periodic, np.mod(x, Lbox), x)

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i]
    ax.scatter(*x[:, idx].T, c='k', s=4**2, ec='none', alpha=0.5)
    ax.set_xlabel(f'{labels[idx[0]]} [Mpc/h]')
    ax.set_ylabel(f'{labels[idx[1]]} [Mpc/h]')
plt.show()